# Exploring Rossmann sales (Phase 2)

This notebook explores the same data as the SQL questions in Phase 1, this time with **pandas** and **Plotly**.
Each section starts with a question, shows one chart, and ends with what the chart tells us.

**How to run it:** open it in VS Code, choose the `.venv` Python as the kernel (top right), then press **Run All**.
MySQL must be running and loaded (`python -m salevora.load_data`).

**One rule for every chart:** summarise first (weekly totals, per-store averages), then plot.
Plotting a million raw rows would make the notebook huge and slow.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

from salevora.db import get_engine

# Show charts in VS Code / Jupyter, and keep a web-viewable copy for nbviewer.
pio.renderers.default = "plotly_mimetype+notebook_connected"
pio.templates.default = "plotly_white"
engine = get_engine()

## 1. Load the data from MySQL

`pd.read_sql` runs a SQL query and returns a DataFrame. MySQL stays the single source of truth;
pandas is where we explore.

In [2]:
history = pd.read_sql("SELECT * FROM sales_history", engine, parse_dates=["sale_date"])
stores = pd.read_sql("SELECT * FROM stores", engine)
print(f"{len(history):,} daily rows from {history.sale_date.min():%d %b %Y} to {history.sale_date.max():%d %b %Y}")
print(f"{len(stores):,} stores")
history.head()

1,017,209 daily rows from 01 Jan 2013 to 31 Jul 2015
1,115 stores


,store_id,sale_date,day_of_week,sales,customers,is_open,promo,state_holiday,school_holiday
0,1,2013-01-01,2,0,0,0,0,a,1
1,1,2013-01-02,3,5530,668,1,0,0,1
2,1,2013-01-03,4,4327,578,1,0,0,1
3,1,2013-01-04,5,4486,619,1,0,0,1
4,1,2013-01-05,6,4997,635,1,0,0,1


## 2. First look: types, blanks and a summary

Always check these three things before drawing anything.

In [3]:
history.dtypes

store_id                  int64
sale_date         datetime64[s]
day_of_week               int64
sales                     int64
customers                 int64
is_open                   int64
promo                     int64
state_holiday               str
school_holiday            int64
dtype: object

In [4]:
# Blank (NULL) values per column. sales_history has none; stores has some by design.
stores.isna().sum()

store_id                          0
store_type                        0
assortment                        0
competition_distance              3
competition_open_since_month    354
competition_open_since_year     354
promo2                            0
promo2_since_week               544
promo2_since_year               544
promo_interval                  544
dtype: int64

In [5]:
# Most charts use open days only: a closed store always has 0 sales.
open_days = history[history.is_open == 1]
print(f"Open store-days: {len(open_days):,} of {len(history):,} ({len(open_days) / len(history):.0%})")
open_days[["sales", "customers"]].describe().round(0)

Open store-days: 844,392 of 1,017,209 (83%)


,sales,customers
count,844392.0,844392.0
mean,6956.0,763.0
std,3104.0,401.0
min,0.0,0.0
25%,4859.0,519.0
50%,6369.0,676.0
75%,8360.0,893.0
max,41551.0,7388.0


## 3. How do total sales move over time?

Weekly totals for the whole chain. The grey band marks July–December 2014, when 180 stores have no data.

In [6]:
weekly = (history.set_index("sale_date")
          .resample("W")
          .agg(total_sales=("sales", "sum"), stores=("store_id", "nunique"))
          .iloc[1:-1])  # drop the two partial weeks at the edges

fig = px.line(weekly, y="total_sales", title="Weekly sales, all stores",
              labels={"sale_date": "Week", "total_sales": "Sales"})
fig.add_vrect(x0="2014-07-01", x1="2014-12-31", fillcolor="grey", opacity=0.2,
              annotation_text="180 stores missing", annotation_position="top left")
fig.show()

In [7]:
weekly.groupby(weekly.index.year).stores.agg(["min", "max"])

,min,max
sale_date,,
2013,1115,1115
2014,935,1115
2015,1115,1115


## 4. What does a typical day's sales look like?

A histogram of sales on open days. `np.histogram` counts the days in each band first,
so the chart only has to draw about 60 bars.

In [8]:
counts, edges = np.histogram(open_days.sales, bins=np.arange(0, 30_001, 500))
bands = pd.DataFrame({"sales_from": edges[:-1], "days": counts})

fig = px.bar(bands, x="sales_from", y="days", title="How many open store-days had each level of sales",
             labels={"sales_from": "Sales (bands of 500)", "days": "Store-days"})
fig.update_traces(marker_line_width=0)
fig.show()

print(f"Mean {open_days.sales.mean():,.0f}, median {open_days.sales.median():,.0f}, "
      f"skew {open_days.sales.skew():.2f}, above 30,000: {(open_days.sales > 30_000).sum()} days")

Mean 6,956, median 6,369, skew 1.59, above 30,000: 153 days


## 5. How do weekday and promotion work together?

The pandas version of SQL question Q4: `groupby` two columns, then average.

In [9]:
by_day = (open_days.groupby(["day_of_week", "promo"]).sales.mean()
          .reset_index()
          .assign(promo=lambda d: d.promo.map({0: "No promo", 1: "Promo"})))
by_day["day"] = by_day.day_of_week.map(dict(enumerate(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"], start=1)))

fig = px.bar(by_day, x="day", y="sales", color="promo", barmode="group",
             title="Average sales per open day, by weekday and promotion",
             labels={"sales": "Average sales", "day": "", "promo": ""})
fig.show()

## 6. Is there a yearly season?

Average sales per open day for each month, one line per year.

In [10]:
monthly = (open_days.assign(year=open_days.sale_date.dt.year, month=open_days.sale_date.dt.month)
           .groupby(["year", "month"]).sales.mean()
           .reset_index())

fig = px.line(monthly, x="month", y="sales", color="year", markers=True,
              title="Average sales per open day, by month",
              labels={"sales": "Average sales", "month": "Month", "year": "Year"})
fig.update_xaxes(dtick=1)
fig.show()

## 7. How different are the store types?

One point per store: its average sales per open day. A box plot shows the middle half of stores (the box),
the median (the line) and unusual stores (the dots).

In [11]:
store_avg = (open_days.groupby("store_id")
             .agg(avg_sales=("sales", "mean"), avg_customers=("customers", "mean"))
             .join(stores.set_index("store_id")[["store_type", "assortment", "competition_distance"]])
             .reset_index())

fig = px.box(store_avg.sort_values("store_type"), x="store_type", y="avg_sales", points="outliers",
             hover_data=["store_id"], title="Average sales per open day, one point per store",
             labels={"store_type": "Store type", "avg_sales": "Average sales"})
fig.show()
store_avg.groupby("store_type").avg_sales.describe()[["count", "50%", "max"]].round(0)

,count,50%,max
store_type,,,
a,602.0,6481.0,21757.0
b,17.0,9745.0,20719.0
c,148.0,6646.0,15531.0
d,348.0,6695.0,18575.0


## 8. Customers and sales: a strong link we can't use

Each point is a store. The colour is the store type.

In [12]:
fig = px.scatter(store_avg, x="avg_customers", y="avg_sales", color="store_type",
                 hover_data=["store_id"], render_mode="svg",
                 category_orders={"store_type": ["a", "b", "c", "d"]},
                 title="Average customers vs average sales, one point per store",
                 labels={"avg_customers": "Average customers per open day", "avg_sales": "Average sales",
                         "store_type": "Store type"})
fig.show()
print(f"Correlation across store-days: {open_days.sales.corr(open_days.customers):.2f}")

Correlation across store-days: 0.82


**Why we can't use it:** the number of customers is only known *after* the day is over.
`test.csv` has no Customers column, so a model that relies on it couldn't forecast the future.
Using it anyway would be **data leakage**.

## 9. Does a nearby competitor matter?

The pandas and Plotly version of SQL question Q6. The x-axis is logarithmic because distances range from 20 m to 76 km.

In [13]:
fig = px.scatter(store_avg.dropna(subset=["competition_distance"]), x="competition_distance", y="avg_sales",
                 log_x=True, opacity=0.6, hover_data=["store_id"], render_mode="svg",
                 title="Distance to the nearest competitor vs average sales, one point per store",
                 labels={"competition_distance": "Distance to competitor (m, log scale)", "avg_sales": "Average sales"})
fig.show()
print(f"Correlation between log distance and average sales: "
      f"{np.log(store_avg.competition_distance).corr(store_avg.avg_sales):.2f}")

Correlation between log distance and average sales: -0.10


## 10. One store up close: smoothing with rolling averages

`rolling(7)` in pandas does the same job as `AVG() OVER (ROWS BETWEEN 6 PRECEDING ...)` in SQL question Q10.
Store 1, last 120 days.

In [14]:
store1 = (history[history.store_id == 1]
          .set_index("sale_date").sales
          .to_frame())
store1["7-day average"] = store1.sales.rolling(7).mean()
store1["14-day average"] = store1.sales.rolling(14).mean()
recent = store1.iloc[-120:].rename(columns={"sales": "Daily sales"})

fig = px.line(recent, title="Store 1: daily sales and rolling averages",
              labels={"sale_date": "", "value": "Sales", "variable": ""})
fig.show()

## Findings

1. **The data is clean but has one big hole.** 1,017,209 rows with no blank values in the daily history;
   83% of store-days are open. Blanks in `stores` are expected (unknown competition dates, Promo2 dates for
   stores without Promo2). The hole: only 935 stores report from July to December 2014 (section 3), which is
   why the weekly total drops there and why the December 2014 peak looks lower than December 2013.
2. **Weekly sales zig-zag every other week.** That's the two-week promotion cycle, visible even in chain totals,
   with a Christmas spike every December.
3. **Daily sales are skewed to the right.** Mean 6,956 vs median 6,369 (skew 1.59), with a long tail up to 41,551.
   In Phase 4 the model will learn `log(sales)` so the few very large days don't dominate.
4. **Promotions lift every weekday, and never run at weekends** (section 5): the pandas result matches SQL Q4.
5. **December is the peak every year**, and 2015 sits above 2014 in 6 of its 7 months (February is the exception),
   matching the +3.2% growth from SQL Q1.
6. **Type b stores are different.** Their median store sells 9,745 per open day vs 6,481–6,695 for the other
   types, and in the customers chart they sit *below* everyone else: many customers, small baskets.
   Type d sits highest: fewer customers who spend more.
7. **Customers predict sales well (correlation 0.82) but can't be a model input:** they're unknown for future days.
8. **Competitor distance barely matters.** The correlation between log distance and average sales is only −0.10,
   slightly favouring stores with a close competitor, as SQL Q6 found.
9. **Rolling averages need the right window.** For store 1 the 7-day average still waves with the promo cycle;
   the 14-day average is smooth. Same lesson as SQL Q10.

**Carry into Phase 4:** day of week, promo, holidays and closures, month, store type and assortment as inputs;
train on log sales; never use customers; handle the July–December 2014 gap.